# Set Up Environment

## Import Libraries

In [1]:
import pandas as pd
import geopandas as gpd

## Set Environment Variables

In [2]:
PROJECT_CRS = "EPSG:3566"

# Input Data

## Preprocessed Forecast Results

In [3]:
forecast_results = pd.read_csv('results/final_forecast_df.csv')
forecast_results[['externalid', 'year', 'final_forecast']]

,externalid,year,final_forecast
0,3601,2027,900
1,3601,2032,1000
2,3601,2036,1100
3,3601,2046,1300
4,3601,2055,1400
...,...,...,...
169,3629,2032,4200
170,3629,2036,4600
171,3629,2046,5400
172,3629,2055,6200


## Traffic Factors

In [4]:
gdf_master_segments = gpd.read_file(
    "zip://data/updated-traffic-factors/Master_Segs_withFactors_20251120.zip"
).to_crs(PROJECT_CRS)

gdf_master_segments

,SEGID,BMP,EMP,DISTANCE,CO_FIPS,PLANAREA,AADT2023,AADT2022,AADT2021,AADT2020,...,FAC_WIN,FAC_SPR,FAC_SUM,FAC_FAL,FAC_MAXMO,FAC_MAX,FACMANADJ,SUTRUCKS,CUTRUCKS,geometry
0,0006_000.0,0.000,0.665,0.666641,27,UDOT,457.0,441.0,474.0,430.0,...,0.8769,1.0071,1.0496,1.0664,10,1.1275,0,0.2496,0.2324,"LINESTRING (916692.404 6835614.744, 920182.189..."
1,0006_000.7,0.665,16.022,15.369839,27,UDOT,457.0,441.0,474.0,430.0,...,0.8769,1.0071,1.0496,1.0664,10,1.1275,0,0.2496,0.2324,"LINESTRING (920182.189 6835168.248, 923166.425..."
2,0006_016.0,16.022,46.017,30.001961,27,UDOT,457.0,441.0,474.0,430.0,...,0.8769,1.0071,1.0496,1.0664,10,1.1275,0,0.2496,0.2324,"LINESTRING (999430.463 6834408.584, 999447.155..."
3,0006_046.0,46.017,60.218,14.194306,27,UDOT,409.0,395.0,424.0,385.0,...,0.8769,1.0071,1.0496,1.0664,10,1.1275,0,0.1751,0.3338,"LINESTRING (1143701.911 6830874.803, 1145024.9..."
4,0006_060.2,60.218,77.545,17.323237,27,UDOT,409.0,395.0,424.0,385.0,...,0.8769,1.0071,1.0496,1.0664,10,1.1275,0,0.1751,0.3338,"LINESTRING (1206604.946 6871212.763, 1206701.0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9252,WFRC_8489,0.000,0.000,0.505514,35,WFRC,0.0,0.0,0.0,0.0,...,0.9411,0.9924,1.0246,1.0419,8,1.0510,0,0.1086,0.0565,"LINESTRING (1518588.41 7388436.516, 1518656.54..."
9253,WFRC_8490,0.000,0.000,0.737896,35,WFRC,0.0,0.0,0.0,0.0,...,0.9411,0.9924,1.0246,1.0419,8,1.0510,0,0.1086,0.0565,"LINESTRING (1521238.95 7388420.585, 1522418.15..."
9254,WFRC_8491,0.000,0.000,0.265495,35,WFRC,0.0,0.0,0.0,0.0,...,0.9411,0.9924,1.0246,1.0419,8,1.0510,0,0.1086,0.0565,"LINESTRING (1525103.396 7388831.353, 1525935.1..."
9255,WFRC_8492,0.000,0.000,0.444725,35,WFRC,0.0,0.0,0.0,0.0,...,0.9519,1.0137,1.0141,1.0203,5,1.0436,0,0.1052,0.0432,"LINESTRING (1527315.01 7447662.776, 1527326.78..."


## Load External-Segment Link

In [5]:
external_segment_link = pd.read_csv("params/externals-segments-link.csv")
external_segment_link

,externalid,segid
0,3601,1082_000.0
1,3602,0013_006.5
2,3603,1112_000.0
3,3604,0015_368.1
4,3605,0038_003.2
5,3606,0091_010.1
6,3607,3462_002.8
7,3608,0039_008.7
8,3609,0084_087.8
9,3610,2688_005.5


# Process Data

## Interpolate Final Forecast Values

In [6]:
# --- 1. Prepare the Full Time Series Index ---
# Create a DataFrame with every externalid for every year (min/max source year).

start_year = forecast_results['year'].min()
end_year = forecast_results['year'].max()

# Create the Cartesian product (all ID x all Year combinations)
df_full_index = pd.MultiIndex.from_product(
    [forecast_results['externalid'].unique(), range(start_year, end_year + 1)],
    names=['externalid', 'year']
).to_frame(index=False).reset_index(drop=True)

In [7]:
# --- 2. Merge Source Data ---
# Join the index to the source data.
df_forecast_interpolated = df_full_index.merge(
    forecast_results,
    on=['externalid', 'year'],
    how='left'
)


In [8]:
# --- 3. Final Assignment and Cleanup ---
# Calculate the interpolation series (Group, Interpolate, Round).
df_forecast_interpolated['interpolated_forecast'] = (
    df_forecast_interpolated.groupby('externalid')['final_forecast']
    .apply(lambda x: x.interpolate(method='linear'))
    .round()
    .astype(int)
).values

In [9]:
df_forecast_interpolated.head(15)

,externalid,year,PROJ_GRP,linear_forecast_notes,segid,linear_forecast,name,external,manual_adj,final_forecast,interpolated_forecast
0,3601,2027,Since 2003,as early as there is data,1082_000.0,890.0,FAR-1082 Bird Refuge,Ext # 3601 - FAR-1082 Bird Refuge,NaN,900.0,900
1,3601,2028,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,920
2,3601,2029,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,940
3,3601,2030,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,960
4,3601,2031,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,980
5,3601,2032,Since 2003,as early as there is data,1082_000.0,989.0,FAR-1082 Bird Refuge,Ext # 3601 - FAR-1082 Bird Refuge,NaN,1000.0,1000
6,3601,2033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1025
7,3601,2034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1050
8,3601,2035,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1075
9,3601,2036,Since 2003,as early as there is data,1082_000.0,1069.0,FAR-1082 Bird Refuge,Ext # 3601 - FAR-1082 Bird Refuge,NaN,1100.0,1100


# Process the Data

In [10]:
df_external_year = pd.read_csv(
    r"archive/v900/external_year_vol.csv"
)

df_external_year

,;Idx_WF,WF_Ext,Year,AWDT,PASS_VOL,TRUCK_MD,TRUCK_HV,AWDT_FAC,AADT,PASSENGER,TRUCK_SU,TRUCK_MU,PctTrk_SU,PctTrk_MU
0,36012010,3601,2010,492,360,66,66,0.956,515,377,69,69,0.135,0.135
1,36012011,3601,2011,487,355,66,66,0.956,510,372,69,69,0.135,0.135
2,36012012,3601,2012,558,408,75,75,0.956,585,427,79,79,0.135,0.135
3,36012013,3601,2013,577,423,77,77,0.956,605,443,81,81,0.135,0.135
4,36012014,3601,2014,587,429,79,79,0.956,615,449,83,83,0.135,0.135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1474,36292056,3629,2056,3408,2446,481,481,0.984,3464,2486,489,489,0.141,0.141
1475,36292057,3629,2057,3457,2481,488,488,0.984,3513,2521,496,496,0.141,0.141
1476,36292058,3629,2058,3505,2515,495,495,0.984,3562,2556,503,503,0.141,0.141
1477,36292059,3629,2059,3554,2550,502,502,0.984,3611,2591,510,510,0.141,0.141


## Initialize the Final Dataframe

In [11]:
# Step 1: Initiate dataframe with the id, and year columns
df_external_year_update = pd.MultiIndex.from_product(
    [
        df_forecast_interpolated['externalid'].unique(), # Unique External IDs
        range(1981, 2061) # Range goes up to, but does not include, 2061
    ],
    names=['WF_Ext', 'Year']
).to_frame(index=False).reset_index(drop=True)

# Generate Idx column
df_external_year_update[';Idx_WF'] = (
    df_external_year_update['WF_Ext'].astype(str) + df_external_year_update['Year'].astype(str)
)

df_external_year_update


,WF_Ext,Year,;Idx_WF
0,3601,1981,36011981
1,3601,1982,36011982
2,3601,1983,36011983
3,3601,1984,36011984
4,3601,1985,36011985
...,...,...,...
2315,3629,2056,36292056
2316,3629,2057,36292057
2317,3629,2058,36292058
2318,3629,2059,36292059


## Segment ID

In [12]:
# Step 2: Join Segment ID from External-Segment Link
df_external_year_update['segid'] = df_external_year_update['WF_Ext'].map(
    external_segment_link.set_index('externalid')['segid']
)

df_external_year_update

,WF_Ext,Year,;Idx_WF,segid
0,3601,1981,36011981,1082_000.0
1,3601,1982,36011982,1082_000.0
2,3601,1983,36011983,1082_000.0
3,3601,1984,36011984,1082_000.0
4,3601,1985,36011985,1082_000.0
...,...,...,...,...
2315,3629,2056,36292056,1826_004.9
2316,3629,2057,36292057,1826_004.9
2317,3629,2058,36292058,1826_004.9
2318,3629,2059,36292059,1826_004.9


## Truck Percentages and Factors

In [13]:
# Step 3: Add Average Weekday Factors, and Truck Percentages
df_external_year_update['PctTrk_SU'] = df_external_year_update['segid'].map(
    gdf_master_segments.set_index('SEGID')['SUTRUCKS'] # Single Unit Truck Percentages
)

df_external_year_update['PctTrk_MU'] = df_external_year_update['segid'].map(
    gdf_master_segments.set_index('SEGID')['CUTRUCKS'] # Combo Unit Truck Percentages
)

df_external_year_update['AWDT_FAC'] = df_external_year_update['segid'].map(
    gdf_master_segments.set_index('SEGID')['FAC_WDAVG'] # Average Weekday Factors
)

df_external_year_update

,WF_Ext,Year,;Idx_WF,segid,PctTrk_SU,PctTrk_MU,AWDT_FAC
0,3601,1981,36011981,1082_000.0,NaN,NaN,NaN
1,3601,1982,36011982,1082_000.0,NaN,NaN,NaN
2,3601,1983,36011983,1082_000.0,NaN,NaN,NaN
3,3601,1984,36011984,1082_000.0,NaN,NaN,NaN
4,3601,1985,36011985,1082_000.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,0.1926,0.1492,1.0404
2316,3629,2057,36292057,1826_004.9,0.1926,0.1492,1.0404
2317,3629,2058,36292058,1826_004.9,0.1926,0.1492,1.0404
2318,3629,2059,36292059,1826_004.9,0.1926,0.1492,1.0404


## Historic AADT (1981 - 2023)

In [14]:
# Step 4: Add Historic AADT Data (1981 - 2023)
df_external_year_update['AADT_Historic'] = (
    gdf_master_segments
    # --- 1. RESHAPE (Convert Wide Data to Long Format) ---
    .melt(
        id_vars=['SEGID'],
        value_vars=[c for c in gdf_master_segments.columns if c.startswith('AADT')],
        var_name='Year_Str',
        value_name='AADT_Historic'
    )
    # --- 2. CLEAN AND PREPARE JOIN KEY ---
    .assign(Year=lambda df: df['Year_Str'].str.replace('AADT', '').astype(int))
    # --- 3. CREATE MAPPING SERIES (The Lookup Table) ---
    .set_index(['SEGID', 'Year'])['AADT_Historic']
    # --- 4. PERFORM LOOKUP AND ALIGNMENT ---
    .reindex(df_external_year_update.set_index(['segid', 'Year']).index)
    # --- 5. ASSIGN FINAL VALUES ---
    .values
)

df_external_year_update

,WF_Ext,Year,;Idx_WF,segid,PctTrk_SU,PctTrk_MU,AWDT_FAC,AADT_Historic
0,3601,1981,36011981,1082_000.0,NaN,NaN,NaN,NaN
1,3601,1982,36011982,1082_000.0,NaN,NaN,NaN,NaN
2,3601,1983,36011983,1082_000.0,NaN,NaN,NaN,NaN
3,3601,1984,36011984,1082_000.0,NaN,NaN,NaN,NaN
4,3601,1985,36011985,1082_000.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,0.1926,0.1492,1.0404,NaN
2316,3629,2057,36292057,1826_004.9,0.1926,0.1492,1.0404,NaN
2317,3629,2058,36292058,1826_004.9,0.1926,0.1492,1.0404,NaN
2318,3629,2059,36292059,1826_004.9,0.1926,0.1492,1.0404,NaN


# Forecasted AADT (2027 - 2060)

In [15]:
# Step 5: Add Forecasted AADT Data (2027 - 2060)
df_external_year_update['AADT_Forecast'] = (
    df_external_year_update.set_index(['WF_Ext', 'Year'])
    .index.map(
        df_forecast_interpolated.set_index(['externalid', 'year'])['interpolated_forecast']
    )
).astype('Int64')

df_external_year_update

,WF_Ext,Year,;Idx_WF,segid,PctTrk_SU,PctTrk_MU,AWDT_FAC,AADT_Historic,AADT_Forecast
0,3601,1981,36011981,1082_000.0,NaN,NaN,NaN,NaN,<NA>
1,3601,1982,36011982,1082_000.0,NaN,NaN,NaN,NaN,<NA>
2,3601,1983,36011983,1082_000.0,NaN,NaN,NaN,NaN,<NA>
3,3601,1984,36011984,1082_000.0,NaN,NaN,NaN,NaN,<NA>
4,3601,1985,36011985,1082_000.0,NaN,NaN,NaN,NaN,<NA>
...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,0.1926,0.1492,1.0404,NaN,6280
2316,3629,2057,36292057,1826_004.9,0.1926,0.1492,1.0404,NaN,6360
2317,3629,2058,36292058,1826_004.9,0.1926,0.1492,1.0404,NaN,6440
2318,3629,2059,36292059,1826_004.9,0.1926,0.1492,1.0404,NaN,6520


## Current AADT (2024-2027)

In [ ]:
# Calculate Current AADT
df_external_year_update['AADT_Current'] = 0

## Combine AADT

In [ ]:
df_external_year_update['AADT'] = (
    df_external_year_update['AADT_Historic']
    .fillna(df_external_year_update['AADT_Forecast'])
    .fillna(df_external_year_update['AADT_Current'])

    .round()
    .astype('Int64')
)

df_external_year_update

,WF_Ext,Year,;Idx_WF,segid,PctTrk_SU,PctTrk_MU,AWDT_FAC,AADT_Historic,AADT_Forecast,AADT
0,3601,1981,36011981,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>
1,3601,1982,36011982,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>
2,3601,1983,36011983,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>
3,3601,1984,36011984,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>
4,3601,1985,36011985,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,0.1926,0.1492,1.0404,NaN,6280,6280
2316,3629,2057,36292057,1826_004.9,0.1926,0.1492,1.0404,NaN,6360,6360
2317,3629,2058,36292058,1826_004.9,0.1926,0.1492,1.0404,NaN,6440,6440
2318,3629,2059,36292059,1826_004.9,0.1926,0.1492,1.0404,NaN,6520,6520


## Passenger, Single Unit Truck, and Combo Unit Truck Volumes

In [18]:
df_external_year_update = df_external_year_update.assign(
    # 1. Calculate and cast the individual truck volumes to nullable integer
    TRUCK_SU=lambda df: (df['AADT'] * df['PctTrk_SU']).round().astype('Int64'),
    TRUCK_MU=lambda df: (df['AADT'] * df['PctTrk_MU']).round().astype('Int64')
).assign(
    # 2. Calculate the passenger volume using the new columns from the first assign() block
    PASSENGER=lambda df: (
        df['AADT'] - (df['TRUCK_SU'] + df['TRUCK_MU'])
    ).round().astype('Int64')
)

df_external_year_update

,WF_Ext,Year,;Idx_WF,segid,PctTrk_SU,PctTrk_MU,AWDT_FAC,AADT_Historic,AADT_Forecast,AADT,TRUCK_SU,TRUCK_MU,PASSENGER
0,3601,1981,36011981,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>
1,3601,1982,36011982,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>
2,3601,1983,36011983,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>
3,3601,1984,36011984,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>
4,3601,1985,36011985,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,0.1926,0.1492,1.0404,NaN,6280,6280,1210,937,4133
2316,3629,2057,36292057,1826_004.9,0.1926,0.1492,1.0404,NaN,6360,6360,1225,949,4186
2317,3629,2058,36292058,1826_004.9,0.1926,0.1492,1.0404,NaN,6440,6440,1240,961,4239
2318,3629,2059,36292059,1826_004.9,0.1926,0.1492,1.0404,NaN,6520,6520,1256,973,4291


## Convert to Average Weekly Number

In [19]:
df_external_year_update = df_external_year_update.assign(
    # 1. Calculate and cast the individual truck volumes to nullable integer
    AWDT=lambda df: (df['AADT'] * df['AWDT_FAC']).round().astype('Int64'),
    PASS_VOL=lambda df: (df['PASSENGER'] * df['AWDT_FAC']).round().astype('Int64'),
    TRUCK_MD=lambda df: (df['TRUCK_SU'] * df['AWDT_FAC']).round().astype('Int64'),
    TRUCK_HV=lambda df: (df['TRUCK_MU'] * df['AWDT_FAC']).round().astype('Int64')
)

df_external_year_update

,WF_Ext,Year,;Idx_WF,segid,PctTrk_SU,PctTrk_MU,AWDT_FAC,AADT_Historic,AADT_Forecast,AADT,TRUCK_SU,TRUCK_MU,PASSENGER,AWDT,PASS_VOL,TRUCK_MD,TRUCK_HV
0,3601,1981,36011981,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,3601,1982,36011982,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,3601,1983,36011983,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,3601,1984,36011984,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,3601,1985,36011985,1082_000.0,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,0.1926,0.1492,1.0404,NaN,6280,6280,1210,937,4133,6534,4300,1259,975
2316,3629,2057,36292057,1826_004.9,0.1926,0.1492,1.0404,NaN,6360,6360,1225,949,4186,6617,4355,1274,987
2317,3629,2058,36292058,1826_004.9,0.1926,0.1492,1.0404,NaN,6440,6440,1240,961,4239,6700,4410,1290,1000
2318,3629,2059,36292059,1826_004.9,0.1926,0.1492,1.0404,NaN,6520,6520,1256,973,4291,6783,4464,1307,1012


# Export Final Results

In [22]:
(
    df_external_year_update

    # 1. Filter columns using the predefined list
    [[
        ';Idx_WF', 'WF_Ext', 'Year',
        'AWDT', 'PASS_VOL', 'TRUCK_MD', 'TRUCK_HV',
        'AWDT_FAC',
        'AADT', 'PASSENGER', 'TRUCK_SU', 'TRUCK_MU',
        'PctTrk_SU', 'PctTrk_MU'
    ]]

    # 2. Filter rows using the boolean mask (Year >= 2010)
    #    We use the boolean mask directly since the preceding step ensures the DataFrame is clean.
    [df_external_year_update['Year'] >= 2010]

    # 3. Export to CSV
    .to_csv('results/external_year_vol.csv', index=False)
)